# Voilà showcase

A tour of what a notebook can do as an app on this stack: layouts, live data, background jobs, uploads, the local model, speech and
your own storage. Everything below is plain Python in **voila-showcase.ipynb**, so open it to see (and copy) how each tab is built.

*Try:* add `?name=YourName` to this page's address, or `?theme=dark` for the dark theme.

In [ ]:
# ── Setup: libraries and a little style ────────────────────────────────────────────────────────────────
import html
import io
import os
import time
from urllib.parse import parse_qs

import boto3
import matplotlib.pyplot as plt
import numpy as np
import openai
import pandas as pd
import ipywidgets as w
from IPython.display import Markdown, clear_output, display

plt.rcParams.update({"figure.figsize": (7, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False,
                     "grid.alpha": 0.3})
BLUE, ORANGE, GREEN = "#2a7de1", "#e1782a", "#2ae17d"


In [ ]:
# ── Deep links: Voilà hands the page's query string to the kernel, so one notebook can serve many views ─────
# (e.g. /voila/render/examples/voila-showcase.ipynb?name=Ada). The value comes from a visitor, so it is escaped before it goes into HTML.
params = {k: v[0] for k, v in parse_qs(os.environ.get("QUERY_STRING", "")).items()}
who = html.escape(params.get("name", "visitor"))

header = w.HTML(f"""
<div style="padding:18px 22px;border-radius:12px;background:linear-gradient(90deg,{BLUE},#a78bfa);color:white">
  <div style="font-size:22px;font-weight:700">Welcome, {who} 👋</div>
  <div style="opacity:.9">Seven small apps in one notebook. Switch tabs; each one is live Python.</div>
</div>""")


In [ ]:
# ── Tab 1 · Layout and widgets ─────────────────────────────────────────────────────────────────────────
# GridspecLayout places widgets on a grid. jslink connects two widgets *in the browser* (no round trip to Python): drag the slider and
# the number box follows instantly. interactive_output re-runs a function in Python whenever any of the listed widgets change.
speed = w.IntSlider(value=50, min=0, max=100, description="Speed")
speed_box = w.BoundedIntText(value=50, min=0, max=100, description="Exact")
w.jslink((speed, "value"), (speed_box, "value"))
colour = w.ColorPicker(value=BLUE, description="Colour")
day = w.DatePicker(description="Date")
mode = w.ToggleButtons(options=["Eco", "Normal", "Turbo"], value="Normal", description="Mode")
tags = w.SelectMultiple(options=["alpha", "beta", "gamma", "delta"], value=["alpha"], rows=4, description="Tags")

grid = w.GridspecLayout(3, 2)
grid[0, 0], grid[0, 1] = speed, speed_box
grid[1, 0], grid[1, 1] = colour, day
grid[2, 0], grid[2, 1] = mode, tags

def summarise(speed, colour, day, mode, tags):
    swatch = f'<span style="display:inline-block;width:14px;height:14px;background:{colour};border-radius:3px;vertical-align:middle"></span>'
    display(w.HTML(f"<b>{mode}</b> mode at <b>{speed}%</b> {swatch} {colour} · date: {day or 'not set'} · tags: {', '.join(tags) or 'none'}"))

summary = w.interactive_output(summarise, {"speed": speed, "colour": colour, "day": day, "mode": mode, "tags": tags})
tab_layout = w.VBox([grid, summary])


In [ ]:
# ── Tab 2 · Data explorer ──────────────────────────────────────────────────────────────────────────────
# Controls filter a pandas DataFrame; an Output widget is a box whose contents we replace each time (a chart plus a table).
rng = np.random.default_rng(7)
df = pd.DataFrame({"region": rng.choice(["North", "South", "East", "West"], 400),
                   "product": rng.choice(["A", "B", "C"], 400),
                   "units": rng.integers(1, 60, 400),
                   "price": rng.normal(20, 4, 400).round(2),
                   "month": rng.integers(1, 13, 400)})
df["revenue"] = (df.units * df.price).round(2)

region = w.Dropdown(options=["All", *sorted(df.region.unique())], description="Region")
months = w.IntRangeSlider(value=[1, 12], min=1, max=12, description="Months", continuous_update=False)
metric = w.RadioButtons(options=["revenue", "units"], description="Metric")
explorer_out = w.Output()

def make_chart():
    d = df[(df.month >= months.value[0]) & (df.month <= months.value[1])]
    if region.value != "All":
        d = d[d.region == region.value]
    fig, ax = plt.subplots()
    d.groupby("product")[metric.value].sum().plot.bar(ax=ax, color=[BLUE, ORANGE, GREEN], rot=0)
    ax.set_ylabel(metric.value)
    ax.set_title(f"{metric.value.title()} by product ({len(d)} rows)")
    return fig, d

def refresh(*_):
    fig, d = make_chart()
    with explorer_out:
        clear_output(wait=True)       # replace the previous chart instead of stacking a new one under it
        display(fig)
        display(d.groupby("region")[["units", "revenue"]].sum().round(0).astype(int))
    plt.close(fig)

for control in (region, months, metric):
    control.observe(refresh, names="value")
refresh()
tab_data = w.VBox([w.HBox([region, metric]), months, explorer_out])


In [ ]:
# ── Tab 3 · A job with live progress ─────────────────────────────────────────────────────────────────────
# A button callback runs in the kernel and can update widgets while it works, so the page shows progress as it happens.
# Each visitor has their own kernel, so two people starting the job do not affect each other.
start = w.Button(description="Start a 5-second job", button_style="primary", icon="play")
bar = w.IntProgress(min=0, max=100, description="Progress", bar_style="info")
status = w.HTML()

def run_job(_):
    start.disabled = True
    bar.value, bar.bar_style = 0, "info"
    for i in range(1, 101):
        time.sleep(0.05)
        bar.value = i
        if i % 25 == 0:
            status.value = f"<b>{i}%</b> · step {i // 25} of 4 finished"
    bar.bar_style = "success"
    status.value = "<b>Done.</b> Real work goes here: train a model, query a database, render a report."
    start.disabled = False

start.on_click(run_job)
tab_job = w.VBox([start, bar, status])


In [ ]:
# ── Tab 4 · Upload your own data ───────────────────────────────────────────────────────────────────────
# FileUpload sends a file from the visitor's browser to the kernel (it stays in memory; nothing is saved unless you write it).
upload = w.FileUpload(accept=".csv", multiple=False, description="Choose a CSV")
sample = w.Button(description="Use sample data", icon="table")
upload_out = w.Output()

def analyse(frame, name):
    with upload_out:
        clear_output(wait=True)
        display(Markdown(f"**{name}**: {len(frame)} rows × {frame.shape[1]} columns"))
        display(frame.head(8))
        numeric = frame.select_dtypes("number")
        if not numeric.empty:
            fig, ax = plt.subplots()
            numeric.iloc[:, 0].plot.hist(ax=ax, bins=20, color=BLUE)
            ax.set_title(f"Distribution of {numeric.columns[0]}")
            display(fig)
            plt.close(fig)

def on_upload(change):
    item = upload.value[0]            # ipywidgets 8: a tuple of dicts with name, size, content ...
    try:
        analyse(pd.read_csv(io.BytesIO(bytes(item["content"]))), item["name"])
    except Exception as exc:          # a bad file must not break the page
        with upload_out:
            clear_output(wait=True)
            print("Could not read that file as CSV:", str(exc)[:150])

upload.observe(on_upload, names="value")
sample.on_click(lambda _: analyse(df.head(100), "sample data"))
tab_upload = w.VBox([w.HBox([upload, sample]), upload_out])


In [ ]:
# ── Tab 5 · Ask the local model, and hear the answer ────────────────────────────────────────────────────
# OPENAI_API_KEY and OPENAI_BASE_URL are already in this environment (your own LiteLLM key, 10 requests a minute), so the standard
# openai client works with no setup. Qwen's thinking is off by default here because it makes replies much faster; the box turns it on.
question = w.Textarea(value="Explain what Voilà is in two sentences.", layout=w.Layout(width="95%", height="80px"))
think = w.Checkbox(value=False, description="Let the model think (slower, better for hard questions)")
ask = w.Button(description="Ask the local model", button_style="primary", icon="comment")
speak = w.Button(description="Read it aloud", icon="volume-up", disabled=True)
answer = w.HTML(layout=w.Layout(border="1px solid #8885", padding="10px", width="95%", min_height="70px"))
audio_out = w.Output()
last = {"text": ""}

def ask_model(_):
    if not os.environ.get("OPENAI_API_KEY"):
        answer.value = "No <code>OPENAI_API_KEY</code> in this environment: start your server from the Hub to get one."
        return
    ask.disabled, speak.disabled = True, True
    answer.value = "<i>waiting for the model…</i>"
    try:
        stream = openai.OpenAI(max_retries=0).chat.completions.create(
            model="agent", stream=True, max_tokens=800,
            messages=[{"role": "user", "content": question.value}],
            extra_body={"chat_template_kwargs": {"enable_thinking": bool(think.value)}})
        text = ""
        for chunk in stream:                                   # tokens arrive as they are generated
            if chunk.choices and chunk.choices[0].delta.content:
                text += chunk.choices[0].delta.content
                answer.value = html.escape(text).replace("\n", "<br>")
        last["text"] = text
        speak.disabled = not text
    except openai.RateLimitError:
        answer.value = "Rate limit reached (10 requests a minute per person). Wait a few seconds and try again."
    except Exception as exc:
        answer.value = "The request failed: " + html.escape(str(exc)[:200])
    finally:
        ask.disabled = False

def read_aloud(_):
    speak.disabled = True
    try:
        wav = openai.OpenAI(max_retries=0).audio.speech.create(model="tts-1", voice="af_heart", response_format="wav",
                                                              input=last["text"][:600]).read()
        with audio_out:
            clear_output(wait=True)
            display(w.Audio(value=wav, format="wav", autoplay=True, controls=True))
    except Exception as exc:
        with audio_out:
            clear_output(wait=True)
            print("Text to speech failed:", str(exc)[:150])
    finally:
        speak.disabled = False

ask.on_click(ask_model)
speak.on_click(read_aloud)
tab_model = w.VBox([question, think, w.HBox([ask, speak]), answer, audio_out])


In [ ]:
# ── Tab 6 · Your storage ──────────────────────────────────────────────────────────────────────────────
# AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY and AWS_ENDPOINT_URL are set for you, so boto3 reaches your own bucket (S3_BUCKET) and the
# shared one. Write results where you can find them again; people can read each other's files in `shared`, and write only under their own name.
bucket = os.environ.get("S3_BUCKET")
shared = os.environ.get("S3_SHARED_BUCKET", "shared")
me = os.environ.get("JUPYTERHUB_USER", "me")
s3 = boto3.client("s3") if os.environ.get("AWS_ACCESS_KEY_ID") else None
listing = w.Output()
saved = w.HTML()
save_mine = w.Button(description="Save the current chart to my bucket", icon="cloud-upload")
save_shared = w.Button(description="Publish it to the shared bucket", icon="share")

def show_listing():
    with listing:
        clear_output(wait=True)
        if s3 is None:
            print("No storage key in this environment: start your server from the Hub to get one.")
            return
        rows = []
        for name in (bucket, shared):
            for obj in s3.list_objects_v2(Bucket=name, MaxKeys=100).get("Contents", []):
                rows.append({"bucket": name, "file": obj["Key"], "KiB": round(obj["Size"] / 1024, 1)})
        display(pd.DataFrame(rows) if rows else Markdown("*Nothing here yet. Save something with a button above.*"))

def upload_chart(target_bucket, key):
    fig, _ = make_chart()                 # the chart from the data explorer tab, with its current filters
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    s3.put_object(Bucket=target_bucket, Key=key, Body=buffer.getvalue(), ContentType="image/png")
    saved.value = f"Saved <code>s3://{html.escape(target_bucket)}/{html.escape(key)}</code>"
    show_listing()

def guarded(action):
    def handler(_):
        if s3 is None:
            saved.value = "No storage key in this environment."
            return
        try:
            action()
        except Exception as exc:
            saved.value = "Could not save: " + html.escape(str(exc)[:200])
    return handler

save_mine.on_click(guarded(lambda: upload_chart(bucket, f"voila-demo/chart-{int(time.time())}.png")))
save_shared.on_click(guarded(lambda: upload_chart(shared, f"{me}/voila-demo-chart.png")))
show_listing()
tab_storage = w.VBox([w.HBox([save_mine, save_shared]), saved, listing])


In [ ]:
# ── Tab 7 · About this page, and putting it all together ─────────────────────────────────────────────────
tab_about = w.HTML("""
<h3 style="margin-top:0">How a notebook becomes an app</h3>
<ul>
  <li><b>One kernel per visitor.</b> Voilà runs the notebook top to bottom for each person, then shows the outputs; the code stays hidden.</li>
  <li><b>Widgets are the interface.</b> A control change goes to Python, Python answers, the page updates. <code>jslink</code> keeps simple links in the browser.</li>
  <li><b>Only you can open this page.</b> The Hub signs people in, and this server belongs to one person. Sharing a dashboard means giving others a copy of the notebook (use the shared bucket) or asking an admin about a shared service.</li>
  <li><b>Page options:</b> <code>?theme=dark</code> or <code>?theme=light</code>, and your own, such as <code>?name=Ada</code>.</li>
  <li><b>Make one yourself:</b> copy <b>voila-basic.ipynb</b>, change it, save, refresh. To run a notebook without a page (a report, a batch), use papermill: <code>papermill in.ipynb out.ipynb -p month 2026-10</code>.</li>
</ul>""")

# ── Put it together ──────────────────────────────────────────────────────────────────────────────────
titles = ["Layout", "Data", "Job", "Upload", "Local model", "Storage", "About"]
tabs = w.Tab(children=[tab_layout, tab_data, tab_job, tab_upload, tab_model, tab_storage, tab_about])
for i, title in enumerate(titles):
    tabs.set_title(i, title)
display(header, tabs)
